# 데이터 준비 — HTP 압축해제 + Quick Draw 다운로드

이 노트북은 **데이터 획득/준비 전용**이다. 학습은 `baseline.ipynb`에서 한다.

| 노트북 | 역할 |
|---|---|
| **`test.ipynb` (이 파일)** | HTP zip 압축해제 · Quick Draw 다운로드 → `data/prepared/` 생성 |
| `baseline.ipynb` | `data/prepared/` 를 읽어 학습 |

## 왜 분리하는가
- 데이터 준비는 **한 번만** 하면 되고 오래 걸린다(압축해제 3.7GB, 다운로드 수백 MB).
  학습 노트북을 재실행할 때마다 반복할 이유가 없다.
- 이 노트북은 **torch/ultralytics가 필요 없다** — GPU 없는 환경에서도 데이터 준비가 가능하다.

## 산출 구조 (JupyterHub / 로컬 동일 보장)
```
data/
├── HTP/                     ← 원본 zip. 읽기 전용, 절대 수정하지 않음
└── prepared/                ← 이 노트북이 생성
    ├── htp/
    │   ├── Training/{집,나무,남자사람,여자사람}/{images,labels}/
    │   └── Validation/{집,나무,남자사람,여자사람}/{images,labels}/
    ├── quickdraw/raw/*.ndjson
    └── manifest.json        ← baseline.ipynb이 읽는 "계약" 파일
```

> **핵심**: 원본 zip 이름은 현재 한글/영문이 혼재하고 계속 바뀌고 있다(`Man.zip` vs `TL_집.zip`).
> 이 노트북은 zip을 **자동 탐색**하고, 출력은 **항상 위 구조·한글 유형명으로 고정**한다.
> 따라서 원본이 어떻게 바뀌어도 `baseline.ipynb`이 보는 구조는 변하지 않는다.


## 1. 환경 세팅

학습용 패키지(torch/ultralytics)는 필요 없다. 준비 작업에 필요한 최소 패키지만 설치한다.


In [ ]:
%pip install -q tqdm pillow pandas


In [ ]:
import os, sys, json, io, zipfile, shutil, urllib.request, time
from pathlib import Path
from urllib.parse import quote
from collections import Counter

from tqdm.auto import tqdm
import pandas as pd

print("Python:", sys.version.split()[0])


## 2. Config 셀  ⚙️

환경이 바뀌면 이 셀만 수정한다.


In [ ]:
# ═══════════════════════ Config ═══════════════════════
def _looks_like_htp(d: Path) -> bool:
    # d 하위 2단계 안에 Training/ + Validation/ 쌍이 있으면 HTP 데이터 루트로 본다
    cands = [d]
    try:
        for p in d.iterdir():
            if p.is_dir():
                cands.append(p)
                cands.extend(q for q in p.iterdir() if q.is_dir())
    except OSError:
        return False
    return any((c / "Training").is_dir() and (c / "Validation").is_dir() for c in cands)

def _find_data_root():
    # 단순히 "존재하는 data/"가 아니라 "실제 HTP 구조를 가진 data/"를 우선한다.
    # (엉뚱한 data 폴더를 잡아 조용히 빈 결과가 나오는 것을 방지)
    here = Path.cwd()
    fallback = None
    for base in [here, *here.parents]:
        for cand in (base / "data", base / "ai" / "models" / "data"):
            if cand.exists():
                if _looks_like_htp(cand):
                    return cand
                if fallback is None:
                    fallback = cand
    return fallback or here / "data"

DATA_ROOT    = _find_data_root()
PREPARED_DIR = DATA_ROOT / "prepared"
HTP_OUT      = PREPARED_DIR / "htp"
QD_OUT       = PREPARED_DIR / "quickdraw" / "raw"
MANIFEST     = PREPARED_DIR / "manifest.json"

HTP_TYPES = ["집", "나무", "남자사람", "여자사람"]   # 출력에 쓰는 고정(정규) 유형명

# ---- 압축해제 옵션 ----
EXTRACT_HTP     = True    # False면 HTP 압축해제 건너뜀
FORCE_EXTRACT   = False   # True면 이미 풀려 있어도 다시 풂
EXTRACT_LIMIT   = None    # 유형·split당 최대 파일 수. None이면 전체(권장). 빠른 점검용: 200

# ---- Quick Draw 옵션 ----
DOWNLOAD_QD     = True
QD_PER_CATEGORY = 3000    # 카테고리당 내려받을 그림 수 (baseline 본학습 기준)
FORCE_DOWNLOAD  = False
QD_BASE_URL     = "https://storage.googleapis.com/quickdraw_dataset/full/simplified/"

# HTP 요소와 대응되는 Quick Draw 카테고리 (존재 확인 완료)
# 미존재로 제외된 것: window, roof, chimney, smoke, hair, sneaker
QD_CATEGORIES = [
    "house", "tree", "door", "fence", "flower", "grass", "bush",
    "sun", "cloud", "moon", "star", "bird", "squirrel", "mountain",
    "face", "eye", "nose", "mouth", "ear", "arm", "hand", "leg", "foot",
    "shoe", "sock", "t-shirt", "pants", "swing set", "ladder", "stairs",
    "river", "pool", "garden",
]

PREPARED_DIR.mkdir(parents=True, exist_ok=True)
print("DATA_ROOT   :", DATA_ROOT)
print("PREPARED_DIR:", PREPARED_DIR.resolve())
print(f"HTP 압축해제={EXTRACT_HTP} (limit={EXTRACT_LIMIT}) / Quick Draw 다운로드={DOWNLOAD_QD} ({QD_PER_CATEGORY}개/카테고리, {len(QD_CATEGORIES)}종)")


## 3. HTP 원본 zip 자동 탐색

폴더·파일 이름이 한글/영문 어느 쪽이든 인식한다.


In [ ]:
def fix_name(name: str) -> str:
    # zip 내부 파일명 유니코드 복원 (환경에 따라 cp437로 깨지는 경우 보정)
    if "�" in name:
        try:
            return name.encode("cp437").decode("utf-8")
        except Exception:
            pass
    return name

def _find_htp_root(data_root: Path):
    # Training/ 과 Validation/ 을 모두 가진 폴더를 하위 2단계까지 탐색
    cands = [data_root]
    if data_root.exists():
        for p in data_root.iterdir():
            if p.is_dir():
                cands.append(p)
                cands.extend(q for q in p.iterdir() if q.is_dir())
    for c in cands:
        if (c / "Training").is_dir() and (c / "Validation").is_dir():
            return c
    return None

def _find_subdir(split_dir: Path, kind: str):
    keys = ("원천", "source", "image", "img") if kind == "src" else ("라벨", "label")
    for p in sorted(split_dir.iterdir()):
        if p.is_dir() and any(k in p.name.lower() for k in keys):
            return p
    return None

def _zip_type(stem: str):
    # 주의: 'woman'이 'man'을 포함하므로 여자사람을 먼저 검사해야 한다.
    s = stem.lower()
    if "여자사람" in s or "woman" in s: return "여자사람"
    if "남자사람" in s or "man"   in s: return "남자사람"
    if "나무"     in s or "tree"  in s: return "나무"
    if "집"       in s or "house" in s: return "집"
    return None

HTP_ROOT = _find_htp_root(DATA_ROOT)
HTP_ZIPS = {}
if HTP_ROOT is not None:
    for _split in ("Training", "Validation"):
        _entry = {}
        for _kind in ("src", "lbl"):
            _sub = _find_subdir(HTP_ROOT / _split, _kind)
            _m = {}
            if _sub is not None:
                for _z in sorted(_sub.glob("*.zip")):
                    _t = _zip_type(_z.stem)
                    if _t:
                        _m[_t] = _z
            _entry[_kind] = _m
        HTP_ZIPS[_split] = _entry

print("HTP_ROOT:", HTP_ROOT)
rows = []
for sp in HTP_ZIPS:
    for t in HTP_TYPES:
        s, l = HTP_ZIPS[sp]["src"].get(t), HTP_ZIPS[sp]["lbl"].get(t)
        rows.append({"split": sp, "유형": t,
                     "원천zip": s.name if s else "❌ 없음",
                     "라벨zip": l.name if l else "❌ 없음"})
display(pd.DataFrame(rows))


## 4. HTP zip 압축해제

### JupyterHub에서도 동일한 구조를 보장하는 방법
zip 내부 멤버명은 `/집_10_남_00137.json` 처럼 **선행 슬래시**가 붙어 있고,
환경(Python 버전·OS)에 따라 한글이 cp437로 깨져 보일 수 있다.
`extractall()`에 맡기면 환경마다 결과가 달라질 여지가 있으므로,
**멤버명에서 파일명만 뽑아 직접 기록**한다. 그러면 어느 환경에서도 결과가 동일하다.

출력 경로는 원본 zip 이름과 무관하게 **항상 한글 정규 유형명**을 쓴다.


In [ ]:
def extract_members(zip_path: Path, out_dir: Path, ext: str, limit=None, desc="") -> int:
    # zip에서 ext 파일을 out_dir에 평탄하게(파일명만) 기록. 반환: 기록한 파일 수
    out_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        members = [m for m in z.namelist() if m.lower().endswith(ext)]
        members.sort(key=lambda m: fix_name(m))     # 결정론적 순서
        if limit:
            members = members[:limit]
        for m in tqdm(members, desc=desc, leave=False):
            fname = Path(fix_name(m)).name          # 선행 '/' 및 디렉터리 성분 제거
            (out_dir / fname).write_bytes(z.read(m))
        return len(members)

def already_done(out_dir: Path, ext: str, expected: int) -> bool:
    if not out_dir.exists():
        return False
    return len(list(out_dir.glob(f"*{ext}"))) >= expected

if not EXTRACT_HTP:
    print("EXTRACT_HTP=False → 건너뜀")
elif HTP_ROOT is None:
    print("⚠️ HTP 원본을 찾지 못했습니다. Config의 DATA_ROOT를 확인하세요.")
else:
    t0 = time.time()
    for split in HTP_ZIPS:
        for htype in HTP_TYPES:
            for kind, ext, sub in (("src", ".jpg", "images"), ("lbl", ".json", "labels")):
                zp = HTP_ZIPS[split][kind].get(htype)
                if zp is None:
                    print(f"  ⚠️ {split}/{htype}/{sub}: zip 없음 — 건너뜀")
                    continue
                out_dir = HTP_OUT / split / htype / sub
                with zipfile.ZipFile(zp) as z:
                    n_expect = sum(1 for m in z.namelist() if m.lower().endswith(ext))
                if EXTRACT_LIMIT:
                    n_expect = min(n_expect, EXTRACT_LIMIT)
                if not FORCE_EXTRACT and already_done(out_dir, ext, n_expect):
                    print(f"  ✓ {split}/{htype}/{sub}: 이미 완료 ({n_expect:,}) — 건너뜀")
                    continue
                n = extract_members(zp, out_dir, ext, EXTRACT_LIMIT,
                                    f"{split}/{htype}/{sub}")
                print(f"  → {split}/{htype}/{sub}: {n:,}개")
    print(f"\n압축해제 완료 ({time.time()-t0:.0f}초)")


In [ ]:
# ── 압축해제 검증: 파일 수 + 이미지↔라벨 stem 매칭 ────────────────
rows, all_ok = [], True
for split in ("Training", "Validation"):
    for htype in HTP_TYPES:
        img_d, lbl_d = HTP_OUT / split / htype / "images", HTP_OUT / split / htype / "labels"
        imgs = {p.stem for p in img_d.glob("*.jpg")} if img_d.exists() else set()
        lbls = {p.stem for p in lbl_d.glob("*.json")} if lbl_d.exists() else set()
        matched = imgs & lbls
        ok = bool(imgs) and imgs == lbls
        all_ok &= ok
        rows.append({"split": split, "유형": htype, "이미지": len(imgs), "라벨": len(lbls),
                     "매칭": len(matched), "이미지만": len(imgs - lbls),
                     "라벨만": len(lbls - imgs), "정상": "✓" if ok else "❌"})
df = pd.DataFrame(rows)
display(df)
print(f"총 이미지 {df['이미지'].sum():,} / 총 라벨 {df['라벨'].sum():,} / 매칭 {df['매칭'].sum():,}")
if all_ok:
    print("전체 정상 ✓")
elif df["이미지"].sum() == 0:
    # 원인을 구분해서 안내한다 (경로 문제를 추출 문제로 오인하지 않도록)
    print("❌ 추출된 파일이 하나도 없습니다 — 압축해제가 아니라 '경로' 문제입니다.")
    print(f"   HTP_ROOT = {HTP_ROOT}")
    print(f"   DATA_ROOT = {DATA_ROOT}")
    print("   → Config 셀에서 DATA_ROOT를 원본 zip이 있는 폴더로 직접 지정하세요.")
else:
    print("⚠️ 이미지/라벨 수가 맞지 않습니다. FORCE_EXTRACT=True로 다시 시도하세요.")


In [ ]:
# ── 육안 확인: 해제된 파일에서 샘플 1건 읽어 bbox 개수 확인 ───────
from PIL import Image
smp_dir = HTP_OUT / "Training" / "집"
if (smp_dir / "images").exists():
    ip = sorted((smp_dir / "images").glob("*.jpg"))[0]
    lp = smp_dir / "labels" / (ip.stem + ".json")
    im = Image.open(ip)
    d = json.loads(lp.read_text(encoding="utf-8"))
    print("샘플     :", ip.name)
    print("이미지    :", im.size, im.mode)
    print("해상도meta:", d["meta"].get("img_resolution"))
    print("class    :", d["annotations"]["class"], "| bbox 수:", len(d["annotations"]["bbox"]))
    print("bbox 예시 :", d["annotations"]["bbox"][:3])


## 5. Quick Draw 다운로드

`.ndjson`은 **줄 단위(JSON Lines)** 라서, 필요한 개수만 읽고 **조기 종료**할 수 있다.
카테고리당 원본은 33~130MB지만 이 방식이면 필요한 만큼만 받는다.

각 줄 구조:
```
{"word":"house", "recognized":true, "drawing":[[[x...],[y...]], ...], ...}
```
`drawing`은 스트로크 목록이며 좌표는 **0–255 정규화 공간**이다.


In [ ]:
def qd_download(category: str, n: int, dest_dir: Path, force=False) -> tuple:
    # 필요한 줄 수만 스트리밍으로 받고 조기 종료. 반환: (경로, 받은 줄 수, 캐시여부)
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest = dest_dir / f"{category.replace(' ', '_')}.ndjson"
    if dest.exists() and not force:
        with dest.open(encoding="utf-8") as f:
            have = sum(1 for _ in f)
        if have >= n:
            return dest, have, True
    url = QD_BASE_URL + quote(category) + ".ndjson"
    req = urllib.request.Request(url, headers={"User-Agent": "htp-dataprep/1.0"})
    lines = []
    with urllib.request.urlopen(req, timeout=180) as r:
        for raw in r:
            lines.append(raw.decode("utf-8"))
            if len(lines) >= n:
                break
    dest.write_text("".join(lines), encoding="utf-8")
    return dest, len(lines), False

if not DOWNLOAD_QD:
    print("DOWNLOAD_QD=False → 건너뜀")
else:
    t0, results = time.time(), []
    for cat in tqdm(QD_CATEGORIES, desc="Quick Draw"):
        try:
            p, n, cached = qd_download(cat, QD_PER_CATEGORY, QD_OUT, FORCE_DOWNLOAD)
            results.append({"카테고리": cat, "줄 수": n,
                            "상태": "캐시" if cached else "다운로드",
                            "MB": round(p.stat().st_size / 1e6, 1)})
        except Exception as e:
            results.append({"카테고리": cat, "줄 수": 0, "상태": f"❌ {type(e).__name__}", "MB": 0})
    qd_df = pd.DataFrame(results)
    display(qd_df)
    print(f"완료 ({time.time()-t0:.0f}초) — 총 {qd_df['MB'].sum():.0f}MB, "
          f"그림 {qd_df['줄 수'].sum():,}개")
    fail = qd_df[qd_df["상태"].str.startswith("❌")]
    if len(fail):
        print("⚠️ 실패한 카테고리가 있습니다:", list(fail["카테고리"]))


In [ ]:
# ── Quick Draw 검증: 파싱 가능 여부 + 좌표 범위 확인 ──────────────
import random
chk = []
for cat in QD_CATEGORIES[:8]:
    p = QD_OUT / f"{cat.replace(' ', '_')}.ndjson"
    if not p.exists():
        continue
    xs_max = ys_max = 0
    n_rec = n_tot = 0
    with p.open(encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= 200: break
            d = json.loads(line)
            n_tot += 1
            n_rec += bool(d.get("recognized", True))
            for s in d["drawing"]:
                xs_max = max(xs_max, max(s[0])); ys_max = max(ys_max, max(s[1]))
    chk.append({"카테고리": cat, "검사": n_tot, "recognized": n_rec,
                "x최대": xs_max, "y최대": ys_max})
display(pd.DataFrame(chk))
print("x/y 최대가 255 이하면 정상 (0–255 정규화 공간).")


## 6. manifest.json — `baseline.ipynb`과의 계약

준비 결과를 요약해 파일로 남긴다. `baseline.ipynb`은 이 파일을 읽어
데이터가 준비됐는지 확인하고 경로를 잡는다.


In [ ]:
manifest = {
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "data_root": str(DATA_ROOT.resolve()),
    "htp": {
        "root": str(HTP_OUT.resolve()),
        "layout": "htp/{split}/{type}/{images|labels}/",
        "types": HTP_TYPES,
        "extract_limit": EXTRACT_LIMIT,
        "counts": {},
    },
    "quickdraw": {
        "root": str(QD_OUT.resolve()),
        "per_category": QD_PER_CATEGORY,
        "categories": {},
    },
}
for split in ("Training", "Validation"):
    manifest["htp"]["counts"][split] = {}
    for htype in HTP_TYPES:
        img_d = HTP_OUT / split / htype / "images"
        lbl_d = HTP_OUT / split / htype / "labels"
        manifest["htp"]["counts"][split][htype] = {
            "images": len(list(img_d.glob("*.jpg"))) if img_d.exists() else 0,
            "labels": len(list(lbl_d.glob("*.json"))) if lbl_d.exists() else 0,
        }
for cat in QD_CATEGORIES:
    p = QD_OUT / f"{cat.replace(' ', '_')}.ndjson"
    if p.exists():
        with p.open(encoding="utf-8") as f:
            manifest["quickdraw"]["categories"][cat] = sum(1 for _ in f)

MANIFEST.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("작성:", MANIFEST.resolve())
tot_img = sum(v["images"] for s in manifest["htp"]["counts"].values() for v in s.values())
print(f"HTP 이미지 {tot_img:,}장 / Quick Draw {len(manifest['quickdraw']['categories'])}종 "
      f"{sum(manifest['quickdraw']['categories'].values()):,}개")


## 7. 정리

### 산출물
| 경로 | 내용 |
|---|---|
| `data/prepared/htp/{split}/{유형}/images/` | HTP 원본 이미지 (jpg, 1280×1280) |
| `data/prepared/htp/{split}/{유형}/labels/` | HTP 어노테이션 (json) |
| `data/prepared/quickdraw/raw/*.ndjson` | Quick Draw 스케치 (0–255 좌표) |
| `data/prepared/manifest.json` | 준비 결과 요약 — `baseline.ipynb`이 읽음 |

### 실행 순서
1. **이 노트북(`test.ipynb`)을 먼저 위에서 아래로 실행** — 데이터 준비
2. 그다음 **`baseline.ipynb` 실행** — 학습

### 재실행 시 주의점
- **원본 zip(`data/HTP/`)은 읽기 전용**으로만 접근한다. 모든 결과는 `data/prepared/` 아래에만 생성된다.
- 이미 완료된 작업은 **자동으로 건너뛴다**(파일 수 비교). 다시 하려면 `FORCE_EXTRACT` /
  `FORCE_DOWNLOAD`를 `True`로 두거나 해당 폴더를 지운다.
- `EXTRACT_LIMIT`을 바꿨다면 `data/prepared/htp/`를 **지우고** 다시 실행해야 반영된다.
- 디스크 사용량: HTP 압축해제 약 **3.7GB**, Quick Draw는 `QD_PER_CATEGORY`에 비례
  (3000개/카테고리 × 33종 ≈ 수백 MB).

### 빠른 점검용 설정
전체를 다 풀기 전에 파이프라인만 확인하려면 Config에서:
```python
EXTRACT_LIMIT   = 200    # 유형·split당 200개만
QD_PER_CATEGORY = 500
```

> **다음 작업 예정**: `baseline.ipynb`이 zip을 직접 읽는 대신 `data/prepared/`를 읽도록 전환하고,
> 범용 손그림 모델과 HTP 전용 모델 **두 가지를 모두 산출·평가**하도록 정리한다.
